# 02 — LoRA & QLoRA Explained

## The problem: full fine-tuning is expensive

A 7B parameter model has ... 7 billion parameters. Training all of them means:
- Huge GPU memory (weights + gradients + optimizer states ≈ **16 bytes per parameter** → ~112 GB for 7B!)
- Slow, expensive, and you need to store a full copy of the model per task

## LoRA (Low-Rank Adaptation)

Key idea: **don't touch the original weights.** Instead, learn a small "patch":

$$W' = W + \Delta W = W + B \cdot A$$

- $W$ is frozen (no gradients!)
- $A$ is a small matrix: $d \times r$
- $B$ is a small matrix: $r \times d$
- $r$ = **rank** (typically 4–64) — much smaller than $d$ (typically 1024–8192)

Example: for $d = 4096$ and $r = 8$:
- Full layer: $4096 \times 4096 \approx 16.7M$ params
- LoRA: $2 \times 4096 \times 8 \approx 65K$ params → **~0.4%** of the original!

### Why does this work?
Research shows the weight *changes* during fine-tuning have low "intrinsic rank" — a small matrix captures almost all of what you need.

In [ ]:
import torch

d, r = 4096, 8

W = torch.randn(d, d)          # frozen original weight
A = torch.randn(d, r)          # trainable LoRA matrix
B = torch.zeros(r, d)          # trainable LoRA matrix (init to zero!)

full_params = d * d
lora_params = d * r + r * d
print(f"Full layer params : {full_params:,}")
print(f"LoRA params      : {lora_params:,}")
print(f"Ratio            : {100 * lora_params / full_params:.2f}%")

# The LoRA "patch" — note B starts at zero, so initially ΔW = 0
delta_W = B @ A
print(f"\nInitial ΔW max abs value: {delta_W.abs().max().item()}")

## QLoRA = LoRA + Quantization

QLoRA goes further: **freeze the base model in 4-bit precision**, then train LoRA adapters on top.

| | Full FT | LoRA | QLoRA |
|---|---|---|---|
| Base model precision | 16-bit | 16-bit | **4-bit** |
| Trainable params | 100% | ~0.1–1% | ~0.1–1% |
| 7B model VRAM | ~112 GB | ~18 GB | **~6 GB** |
| Runs on | Multi-GPU cluster | Gaming GPU | Free Colab T4 |

### Key QLoRA tricks
1. **NF4** (NormalFloat4) — a 4-bit format designed for normally-distributed weights
2. **Double quantization** — quantize the quantization constants too
3. **Paged optimizers** — offload optimizer state to CPU RAM when memory spikes

## Key hyperparameters you'll actually tune

| Parameter | What it does | Typical values |
|-----------|--------------|----------------|
| `lora_rank` (r) | Capacity of the adapter | 8 (simple tasks) – 64+ (complex) |
| `lora_alpha` | Scaling of ΔW (effective scale = α/r) | 16–32, often 2×r |
| `lora_target` | Which layers get LoRA | `all` (best quality) or `q_proj,v_proj` (fastest) |
| `learning_rate` | How fast adapters learn | 1e-4 – 5e-5 for LoRA |
| `num_train_epochs` | Passes over the data | 1–3 (more = overfitting risk!) |

> **Tip:** start with `rank=8, alpha=16, lr=5e-5, 3 epochs` — a great default for "Hello World" tasks.

## Merging: one model at the end

At inference you can either keep the adapter separate (load base + patch) or **merge** them:

$$W_{merged} = W + \frac{\alpha}{r} \cdot B A$$

Merged models have zero inference overhead — perfect for deployment.

## Summary

- **LoRA**: train tiny low-rank matrices instead of full weights
- **QLoRA**: do that on top of a 4-bit quantized model → fine-tune on a laptop GPU
- Adapters are small files (~10–100 MB) you can swap per task

**Next:** `03_finetune_with_llama_factory.ipynb` — let's actually train something! 🚀